# Habitat Analyses with geeo: Worked Examples

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonsnill/geeo/blob/master/docs/habitat-type-mapping_geeo.ipynb)

This tutorial walks through the three worked examples demonstrating how GEEO can be used in ecological research. The overarching theme of these examples is to illustrate how to generate satellite image features for habitat analyses:

1. **Example 1: Retrieving Habitat Metrics from Multi-Temporal Landsat Imagery** - Replicating and extending the multi-seasonal Spectral-Temporal-Metric (STM) and quality-screening (NVO) workflow of Oeser et al. (2019) using Landsat-8 and -9 data.
2. **Example 2: Equidistant, Interpolated Time Series** - Generating a regular 16-day time series across 15 years (2010–2024) from the Landsat archive and Radial Basis Function convolution.
3. **Example 3: Exporting imagery for a reference land cover survey** - Extracting analysis-ready time series stacks (TSS) at 505,509 pan-European LUCAS survey points, enabling wall-to-wall mapping of land cover.

## 0. Workflow Design & Use

A brief overview of **geeo**'s fundamental workflow design and use.

**geeo** is a processing pipeline and collection of algorithms for obtaining Analysis-Ready-Data (ARD) and higher-level image products from multispectral image archives, primarily the Landsat and Sentinel-2 archives, using the Google Earth Engine Python API. 

*geeo* follows a modular structure, organised along hierarchical processing levels, and corresponding image products (Figure below, left):

- LEVEL-2 (`geeo/level2`): Preprocessed, harmonized, spatio-temporally subsetted surface reflectance time series stacks (TSS) and mosaics (TSM)   
- LEVEL-3 (`geeo/level3`): Advanced spectral features including Spectral-Temporal-Metrics (STM) and Pixel-based Composites (PBC), as well as Time Series Interpolation (TSI)
- LEVEL-4 (`geeo/level4`): Semantic features, i.e. information derived from spectral signals. Currently implemented in this workflow are Land Surface Phenology (LSP) metrics.   
- EXPORT (`geeo/export`): Export module handling metadata and projection settings, and constructing export tasks.

<div>
<img src="https://raw.githubusercontent.com/leonsnill/geeo/develop/geeo/data/fig/geeo_workflow_manuscript.png" width="50%" style="display:block; margin: 0 auto;"/>
</div>

Typically, there is no need to directly interact with the submodules as described above, users simply provide the processing instructions via a parameter .yml-file or python dictionary. The intended approach to using *geeo* is to:

1. **Create a .yml parameter file** from an existing blueprint using the `create_parameter_file()` function
2. **Adjust the parameter settings** in the .yml file to your requirements
3. **Run the adjusted parameter file** using `run_param()` to trigger the processing (and export)

Users can use *geeo* also "interactively" and/or within their own workflows by simply accessing the returned dictionary (and containing data) after *geeo* processed the instructions.

## 1. Setup & Environment

This tutorial assumes that you have already installed *geeo* and authenticated your Google Earth Engine session. For installation instructions and an overview of the core architecture, please refer to the introductory tutorial (	utorial_0_introducing-geeo.ipynb) and the [geeo GitHub repository](https://github.com/leonsnill/geeo).

We begin by importing the required packages:

In [ ]:
my_project_name = 'eexnill'  # change this to your Google Cloud project name that has access to Earth Engine

# earth engine python API
import ee
ee.Authenticate()
ee.Initialize(project=my_project_name)
# eerepr adds an interactive html representation of Earth Engine objects in Jupyter notebooks
import eerepr
eerepr.initialize()
# geemap allows interactive mapping with Earth Engine in Python
import geemap

# GEEO
try:  # if geeo is already installed in environment
    import geeo
except:   # not installed, install it
    !pip install --quiet git+https://github.com/leonsnill/geeo.git
    import geeo

> **Note on interactive execution:** In the code cells below, large export tasks (EXPORT_IMAGE: False, EXPORT_TABLE: False) are disabled by default to avoid triggering long-running server exports during tutorial runs. Precomputed products and visualisations are loaded from precomputed Earth Engine assets so you can inspect the outputs immediately and we avoid unnecessary ressource use.

## 2. Study Area & Spatial Extent

Our primary study area is a 150 × 150 km region at the French-Italian border of the southwestern Alps. Spanning an elevation gradient from lowland valleys to high-alpine peaks, it encompasses diverse biome types including temperate deciduous and coniferous forests, Mediterranean shrublands, and alpine grasslands.

**geeo** provides the helper function create_roi() to convert local vector files (e.g. GeoPackage, Shapefile, GeoJSON) into server-side Earth Engine objects with consistent spatial metadata:

In [ ]:
study_area = 'https://raw.githubusercontent.com/leonsnill/geeo/master/geeo/data/GLANCE_EU_150-X029-Y029.gpkg'  # the study area is saved in GEEO's GitHub repo
roi = geeo.create_roi(study_area)  # returns a dictionary of various formats, including an ee.Geometry
roi.keys()

create_roi() returns a dictionary containing the spatial region represented in several formats:
- roi_geom: An [ee.Geometry](https://developers.google.com/earth-engine/guides/geometries) representing the polygon boundary.
- roi_featcol: An [ee.FeatureCollection](https://developers.google.com/earth-engine/guides/feature_collections) wrapping the geometry with metadata.
- roi_bbox: An ee.Geometry bounding box of the spatial extent.
- roi_gdf: A local GeoPandas [GeoDataFrame](https://geopandas.org/en/stable/docs/reference/geodataframe.html).


The ee.FeatureCollection defines the 150 × 150 km bounding tile used for the processing workflows below. We visualise the study area extent using geemap:

In [ ]:
roi_featcol = roi['roi_featcol']  # extract the ee.FeatureCollection

M = geemap.Map(center=[44, 7], zoom=7)
M.add_basemap('HYBRID')
vis_params = {'color': 'red', 'width': 2, 'lineType': 'solid','fillColor': '00000000'}
M.addLayer(roi_featcol.style(**vis_params), {}, 'Study Area')
M

---

## Example 1: Retrieving Habitat Metrics from Multi-Temporal Landsat Imagery

Multispectral satellite time series capture vegetation productivity, moisture dynamics, and phenological seasonality - usefull predictors for species distribution models (SDMs) and habitat suitability assessments. Widely used frameworks, such as the Dynamic Habitat Indices (DHIs; Radeloff et al., 2019; Razenkova et al., 2025) and multitemporal habitat metrics (Oeser et al., 2019), characterise habitat conditions using intra-annual observations of remotely sensed indicators.

In remote sensing, these metrics are derived as **Spectral Temporal Metrics (STMs)**. Spectral Temporal Metrics are summary statistics (percentiles, mean, standard deviation) across valid observations for defined temporal intervals. Spectral Temporal Metrics serve two purposes:
1. **Dimensionality reduction:** They condense multi-year, high-dimensional observation stacks into a compact set of ecologically meaningful predictors representing median conditions, variability, and extremes.
2. **Gap handling:** Optical satellite observations are regularly obscured by clouds, shadows, and snow. By aggregating over temporal windows, Spectral Temporal Metrics bridge quality-masked gaps to deliver continuous, wall-to-wall feature surfaces.

### Workflow Alignment with Oeser et al. (2019)
Following the methodology of Oeser et al. (2019) for wildlife habitat mapping, we configure *geeo* to derive multi-seasonal metrics across our Alpine study area:
- **Sensor archive:** Landsat 8 and Landsat 9 Collection 2 Level-2 surface reflectance, screened for clear observations (< 75% scene cloud cover, cloud and shadow masking via QA bands).
- **Target period:** Three-year baseline (2022–2024).
- **Temporal folding:** Four seasons—Spring (March–May), Summer (June–August), Autumn (September–November), and Winter (December–February).
- **Spectral indices:** Tasseled Cap Greenness (TCG; photosynthetic activity), Brightness (TCB; bare soil / openness), and Wetness (TCW; vegetation and soil moisture).
- **Reducers:** 10th percentile (p10), median (p50), 90th percentile (p90), and standard deviation (stdDev).
- **Quality screening:** Number of Valid Observations (NVO), tracking clear-pixel availability across each season to evaluate metric reliability.

### Specifying Processing Settings

As described above (0. Workflow Design & Use), workflows in *geeo* are parameterised either through a YAML configuration file or directly through Python dictionaries. We have parameterised the file [example-1_habitat-metrics.yml](habitat-type-mapping\example-1_habitat-metrics.yml) to contain the desired processing settings from the above list. You can inspect the file to get familiar with the parameters. We provide an extensive overview for all parameters in the [Documentation](../documentation.md), but for now, we focus on the paramaters required for this example.

To give you an overview of the adjusted parameters, we can print the parameters:

In [ ]:
# load parameters from the .yml file into a dictionary
prm_habitat_metrics = geeo.load_parameters('https://raw.githubusercontent.com/leonsnill/geeo/develop/docs/tutorials/habitat-type-mapping/example-1_habitat-metrics.yml')

# we will print the parameters and associated values for illustration
new_parameters_of_interest = [
    'YEAR_MIN', 
    'YEAR_MAX',
    'MONTH_MIN',
    'MONTH_MAX',
    'ROI',
    'SENSORS',
    'MAX_CLOUD', 
    'MASKS_LANDSAT',
    'FEATURES',
    'FOLD_CUSTOM',
    'NVO',
    'NVO_FOLDING',
    'STM',
    'STM_BASE_IMGCOL',
    'STM_FOLDING'
]

for p in new_parameters_of_interest:
    print(f"{p}: {prm_habitat_metrics[p]}")

Alternatively, we could specify the settings directly in a python dictionary, but this requires being familiar with *geeo* and its parameters, so that the parameter file approach is the easiest and intended way to instruct *geeo*. For completeness, this would look as follows:

In [ ]:
prm_habitat_metrics = {
    'YEAR_MIN': 2022,
    'YEAR_MAX': 2024,
    'MONTH_MIN': 1,                                 # we include all months at first, the seasonal subwindows are defined in the folding section
    'MONTH_MAX': 12,
    'ROI': [6.26, 44.35, 6.57, 44.58],              # note that we defined a smaller ROI than "study area" in case exporting is triggered to avoid unnecessary ressource use!
    'SENSORS': ['L8', 'L9'],                        # Landsat-8 and Landsat-9 Collection 2 Tier 1 surface reflectance data
    'MAX_CLOUD': 75,                                # only include scenes with less or equal 75% cloud cover
    'MASKS_LANDSAT': [                              # mask all pixels of this kind
        'cloud', 'cshadow', 
        'fill', 'dilated'
        ],
    'FEATURES': ['TCG', 'TCB', 'TCW'],              # Tasseled Cap greenness, brightness, and wetness
    'FOLD_CUSTOM': {                                # Temporal folding allows to calculate habitat metrics for custom time windows, e.g. month ranges
        'month': ['3-5', '6-8', '9-11', '12-2'],    # instead, we want a custom range of months
        },
    'NVO': True,                                    # Number of Valid Observations (NVO) is a quality metric that measures data availability for each pixel over time
    'NVO_FOLDING': True,                            # we want to calculate the quality layer for each of the custom time windows defined in the folding section
    'STM': ['p10', 'p50', 'p90', 'stdDev'],         # statistical metrics/reducers: 10th, 50th, 90th percentiles, and the standard deviation
    'STM_BASE_IMGCOL': 'TSS',                       # collection on which to calculate the STMs; here we use the basic Time-Series-Stack (TSS)
    'STM_FOLDING': True,                            # actually apply the custom folding settings when calculating the STMs
}

> **Note:** Using YAML parameter files is generally recommended as it separates configuration from code and ensures reproducibility. Here we use an in-line dictionary to make every setting explicit within the notebook.

### Executing the Pipeline with run_param()

Once we have the settings in place, we can run the processing using the `run_param()` function:

In [ ]:
run_habitat_metrics = geeo.run_param(prm_habitat_metrics)

As described above (0. Workflow Desing & Use), `run_param` is a wrapper function that executes the entire chain of `run_level2()` -> `run_level3()` -> `run_level4` -> `run_export()`. Each module receives the parameter dictionary, validates the parameters, attaches the resulting Earth Engine objects (e.g. ee.ImageCollection), and passes the enriched dictionary to the subsequent step. Our variable `run_habitat_metrics` contains the full execution and results of the pipeline, i.e. user-defined parameters merged with geeo's system defaults (parameters not explicitly modified by the user), alongside references to the processed server-side images / collections.

Key outputs in our example are:

- Time Series Stack (`TSS`): The analysis-ready stack of imagery over time (Level-2)
- Spectral Temporal Metrics (`STM`): The image collection containing the metrics for each of our seasonal time windows (Level-3)
- Number of Valid Obersvations (`NVO`): The image containing four bands with the valid pixel count for each of our seasonal time window (Level-3)

### Inspecting the Products (STMs and NVO)

The 'STM' key holds an ee.ImageCollection containing one multi-band ee.Image for each of the four seasonal windows:

In [ ]:
STM = run_habitat_metrics['STM']  # the calculated metrics are stored in an ee.ImageCollection at the STM key
STM  # print STM ee.ImageCollection object and inspect its properties

Expanding the ee.ImageCollection in the Earth Engine representation widget reveals:
- **4 Images:** Corresponding to the four defined seasonal folds.
- **12 Bands per Image:** 3 spectral indices (TCG, TCB, TCW) $\times$ 4 statistical reducers (p10, p50, p90, stdDev).
- **Standardised Temporal Metadata:** Each image's system:index explicitly documents the temporal aggregation window (e.g., Y2022-2024_M03-05_D001-366). When exporting, *geeo* automatically embeds these temporal identifiers into band names and output filenames.

### Configuring Export Settings & Asset Retrieval

Exporting raster collections in Earth Engine requires setting Coordinate Reference Systems (CRS), spatial bounds, and pixel resolutions. Unspecified metadata can lead to unintended grid offsets and reprojection artifacts.

**geeo** facilitates exact spatial extent and pixel alignment via exlicitly defining the CRS transform and image dimensions instead of relying on Earth Engine to figure this out based on fewer implicit settings (usually users only set a 'scale', i.e. pixel grain size and 'region', i.e. a study area extent). geeo also allows to use predefined (continental) gridding schemes such as the GLANCE continental projection system.

Key parameters for our export to the desired study area are:
- CRS: Target coordinate system ('EU' for European LAEA projection from the GLANCE system).
- SCALE: 30 m spatial resolution matching Landsat ground sampling distance.
- RESAMPLING: Bilinear interpolation for continuous spectral data.
- DATA_TYPE: 16-bit signed integer (int16) with a scale factor of 10,000 to efficiently store the data, with no-data values assigned to -9999.
- EXPORT_LOCATION: Target destination ('Asset' or 'Drive').

In [ ]:
habitat_metric_settings_export = {
    'PIX_RES': 30,  # spatial resolution
    'CRS': 'EU',  # EU: Europe glance (https://measures-glance.github.io/glance-grids/)
    'RESAMPLING_METHOD': 'bilinear',  # bilinear resampling for continuous data
    'DATATYPE': 'int16',  # output data type ...
    'DATATYPE_SCALE': 10000,   # ... and scale factor to convert to integer
    'NODATA_VALUE': -9999,  # output nodata value
    'EXPORT_IMAGE': False,  # whether to export the images: setting to true would trigger the export
    'EXPORT_STM': True,  # whether to export the calculated STMs
    'EXPORT_NVO': True,  # whether to export the calculated NVOs
    'EXPORT_LOCATION': 'Asset',  # 'Drive' or 'Asset'
    'EXPORT_DIRECTORY': 'projects/eexnill/assets/geeo_public',  # path to export location
    'EXPORT_DESC': 'HABITAT_METRICS'  # description appended to each image name
}

prm_habitat_metrics.update(habitat_metric_settings_export)

run_habitat_metrics = geeo.run_param(prm_habitat_metrics)  # !!! note that EXPORT_IMAGE is intentionally set to False above here to not trigger unintended exports

> **Note:** EXPORT_IMAGE is set to False above to avoid triggering duplicate export tasks. The exported asset from this run has been precomputed and made publicly accessible for inspection below.

### Visualizing the Products (STMs and NVO)

Once the export task completes on Earth Engine, the multi-band Spectral Temporal Metric is stored under the specified asset path (projects/eexnill/assets/geeo_public/STM_HABITAT_METRICS_2022-2024_LSAT). We can load and inspect the asset:

In [ ]:
img_habitat = ee.Image('projects/eexnill/assets/geeo_public/STM_HABITAT_METRICS_2022-2024_LSAT')
print(img_habitat.bandNames().getInfo()[:5])  # print the first 5 band names

We visualise the exported STM asset with geemap:

In [ ]:
M = geemap.Map(center=[44.25, 7], zoom=8)
M.add_basemap('HYBRID')
selected_bands = [
    'Y2022-2024_M06-08_D001-366_TCB_p50',
    'Y2022-2024_M06-08_D001-366_TCG_p50',
    'Y2022-2024_M06-08_D001-366_TCW_p50'
]
M.addLayer(img_habitat, {'bands': selected_bands, 'min': [1000, 0, -2000], 'max': [6000, 3000, 0]}, 'Summer median STM')
M.addLayer(roi_featcol.style(**vis_params), {}, 'Study Area')
M

Second, we add the Number of Valid Observations (NVO) layer () that allows us to assess the underlying data availability per pixel for the given time windows and thus serves as an important layer for checking the robustness and reliability of our image products:

In [ ]:
img_nvo = ee.Image('projects/eexnill/assets/geeo_public/NVO_HABITAT_METRICS_2022-2024_LSAT')
print(img_nvo.bandNames().getInfo())  # print the first 5 band names

In [ ]:
M.addLayer(img_nvo.select([1]), {'min': 2, 'max': 50, 'palette': 'gnuplot2'}, 'Number of Valid Observations (NVO) for summer 2022-2024')
M

---

## Example 2: Equidistant, Interpolated Time Series

### Background: Time-Series Interpolation & Daily Mosaicking

While seasonal STMs provide static temporal aggregates, many ecological investigations require dynamic time series. Many applications use interpolated spectral time series, including, for example, assessments of land-use intensity (Rufin et al., 2019), characterisiation of land surface phenology (Kowalski et al., 2020), or for differentiating phenologically complex vegetation types such as in heterogenous savanna ecosystems (Harkort et al. 2025). 

However, observation intervals vary due to factors including satellite orbits and cloud cover, hence optical archives (Landsat and Sentinel-2) are irregular: To resolve this, **Time Series Interpolation (TSI)** reconstructs an equidistant, (potentially) gap-free time series by convolving irregular observations over moving temporal windows.

#### Radial Basis Function (RBF) Convolution
*geeo* implements convolution-based temporal interpolation using Gaussian Radial Basis Functions (RBF). Rather than fitting rigid parametric curve models (e.g., asymmetric double logistics), RBF interpolation evaluates a moving Gaussian kernel across target timestamps:

$$
\hat{y}(t_0) = \frac{\sum_{i=1}^{N} w_i y_i}{\sum_{i=1}^{N} w_i}
$$

where:
- $\hat{y}(t_0)$ is the interpolated value at target time $t_0$.
- $y_i$ is the observed value (e.g., surface reflectance or index) at time $t_i$.
- $N$ is the number of observations within the temporal window.
- $\sum_{i=1}^{N} w_i$ normalizes the sum so the weights sum to 1.
- $w_i$ is the Gaussian weight assigned to observation $i$:

$$
w_i = \exp\left(-\frac{(t_i - t_0)^2}{2\sigma_t^2}\right)
$$

where:
- $w_i$ is the weight assigned to observation $i$.
- $t_i$ is the acquisition time of observation $i$.
- $t_0$ is the target interpolation time.
- $\sigma_t$ is the temporal standard deviation (bandwidth) controlling how rapidly the weights decay with temporal distance.

#### Resolving Data Density Variations with Multi-Kernel Weighting
A single kernel with a narrow width ($\sigma = 16\text{ days}$) accurately tracks rapid phenological transitions during data-dense periods, but can result in gaps during extended nodata periods. Conversely, a wide kernel smoothes persistent gaps but can also oversmooth actual observations and true dynamics. 

*geeo* addresses this trade-off by offering the methods detailed in Schwieder et al. (2016) and Frantz (2019):
- **1RBF**: Single Gaussian kernel with user-defined standard deviation ($\sigma_1$) and search window ($\text{WIN}_1$).
- **2RBF / 3RBF**: Dual- or triple-kernel interpolation. Here, narrow kernels are prioritised when valid observations are abundant (controlled by threshold $\text{BW}_1$), while wider fallback kernels activate automatically when data availability drops.

### Parameterising 1RBF Interpolation

In this example, we generate an equidistant 16-day NDVI time series across 15 years (2010–2024) for the 150 × 150 km Alpine study area using the combined Landsat 7, 8, and 9 archives.

We load the parameters from the parameter file (example-2_time-series-interpolation.yml) using geeo.read_param():

In [ ]:
# load parameters from the .yml file into a dictionary
prm_tsi = geeo.load_parameters('https://raw.githubusercontent.com/leonsnill/geeo/develop/docs/tutorials/habitat-type-mapping/example-2_time-series-interpolation.yml')

# we will print the parameters and associated values for illustration
new_parameters_of_interest = [
    'TSI',
    'TSI_BASE_IMGCOL',
    'INTERVAL',
    'INTERVAL_UNIT',
    'INIT_JANUARY_1ST',
    'SIGMA1',
    'WIN1',
]

for p in new_parameters_of_interest:
    print(f"{p}: {prm_tsi[p]}")

The loaded configuration parameters specify:
- **Sensors:** Combined Landsat constellation (SENSORS: ['L9', 'L8', 'L7']) spanning 2010 to 2024.
- **Base Collection:** Time Series Mosaic (TSM) (TSI_BASE_IMGCOL: 'TSM') to consolidate overlapping same-day acquisitions into single daily observations.
- **Interpolation Scheme:** Single Gaussian kernel (TSI: '1RBF') with a standard deviation of 16 days (SIGMA1: 16) and an inclusion half-window of 64 days (WIN1: 64).
- **Temporal Grid:** 16-day intervals (INTERVAL: 16, INTERVAL_UNIT: 'day') anchored to January 1st (INIT_JANUARY_1ST: True).
- **Spectral Target:** Normalized Difference Vegetation Index (FEATURES: ['NDVI']).

### Evaluating Interpolated Time Series at Reference Sites

To assess how well the 1RBF interpolation captures true vegetation dynamics and suppresses residual noise, we evaluate the resulting time series at two contrasting ecological reference sites within the study area:
1. **Alpine Grassland:** Characterised by prolonged winter snow cover (near-zero NDVI) followed by a sharp summer green-up peak.
2. **Subalpine Forest–Grassland Ecotone:** Characterised by higher baseline NDVI with moderate seasonal amplitude.

#### Site A: Alpine Grassland

In [ ]:
from geeo import plot_tsi

alpine_grassland = [6.453384, 44.44164]
forest_grassland_ecotone = [6.460557, 44.428797]

fig, ax = plot_tsi(prm_tsi, roi=alpine_grassland, feature='NDVI', show_masked=True, figsize=(9, 3),
                              color='orangered', title='Alpine grassland', ylim=[-0.2, 1])

#### Site B: Subalpine Forest–Grassland Ecotone

In [ ]:
fig, ax = plot_tsi(prm_tsi, roi=forest_grassland_ecotone, feature='NDVI', show_masked=True, figsize=(9, 3),
                              color='blue', title='Forest-grassland ecotone', ylim=[-0.2, 1])

### Running the Interpolation Pipeline

We pass this configuration to geeo.run_param(). As before, EXPORT_IMAGE is set to False in the snippet below to avoid triggering server tasks; the resulting asset has been precomputed for evaluation:

In [ ]:
my_run = geeo.run_param(prm_tsi)  # runs the analysis as specified in the .yml file

Once processed and exported, the 15-year interpolated multi-band NDVI asset is available in Earth Engine:

In [ ]:
tsi_ndvi = ee.Image('projects/eexnill/assets/geeo_public/landsat-tsi/TSI_HABITAT_EU_150-X029-Y029_2010-2024_LSAT_NDVI')

Inspecting the band names confirms the regular 16-day spacing from 2010 to 2024:

In [ ]:
tsi_ndvi.bandNames()

In [ ]:
# visualize TSI
example_point = ee.Geometry.Point([6.453384, 44.44164])

M = geemap.Map(center=[44.25, 7], zoom=8)
M.add_basemap('HYBRID')
selected_bands = ['NDVI_20240328', 'NDVI_20240616', 'NDVI_20240904']
M.addLayer(tsi_ndvi, {'bands': selected_bands, 'min': 0, 'max': 8000}, 'TSI NDVI')
M.addLayer(example_point, {'color': 'red'}, 'Example Point')
M.addLayer(roi_featcol.style(**vis_params), {}, 'Study Area')
M

---

## Example 3: Extracting Reference Point Time Series & Combining Server-Side with Local Analysis

In this example we want to showcase how users can export image products for geospatial reference locations to a table format. Specifically, we illustrate this in the context of large-area land cover mapping analogously to the methodology described by Pflugmacher et al., (2019). In their paper, the authors use the LUCAS 'Land use and land cover' survey reference dataset in combination with multispectral Landsat-8 data to map pan-European land cover for the year 2015. 

This example relates to a common aspect in ecological or biogeographic modelling, including habitat mapping and species distribution modelling, in which for model parameterisation and/or validation, instead of creating area-wide predictor imagery, only obtaining data for available reference points (e.g. occurrence records) is required. Only after the models have been build and evaluated, users "know" which image features they actually need to map their target using a "final" model. By only obtaining the relevant data at first instead of calculating and downloading all possibly relevant imagery a priori to modelling, users save (often substantial) computational and storage resources. Additionaly, server-side memory limits and execution timeouts in Earth Engine often also require the users to process data more efficiently, e.g. by using spatial tiling and temporal filtering. 

This primary goal of this example is to showcase how *geeo* can be used to ... 

1. ... extract satellite-derived image products at geospatial reference samples, and how this can be achieved in a spatially structured way using geeo`s tiling functions.

The example does nevertheless cover the whole workflow that follows, illustrating how *geeo* integrates into a larger workflow, including also ...

2. ... local conversion of the image data to image predictors, as well as building and evaluating a model for land cover mapping
3. ... retrieval of the image features and prediction of land cover inside Earth Engine

### 1) Exporting image data for reference samples

Our reference samples for mapping land cover from the come **Land Use/Cover Area frame Survey (LUCAS)** that provides in-situ field observations across European Union member states. We utilise the harmonised LUCAS dataset compiled by d'Andrimont et al. (2020), which standardizes surveys from 2006 to 2018 into a unified classification scheme. Following Pflugmacher et al. (2019), we group observations into a total of 12 land cover classes:

| Main Class | Description |
| :--- | :--- |
| **Artificial land** | Impervious surfaces (buildings, roads, artificial structures). |
| **Cropland, seasonal** | Annual arable crops (cereals, root crops, vegetables, fodder). |
| **Cropland, perennial** | Permanent woody crops (orchards, vineyards, olive groves). |
| **Forest, broadleaved** | Woodland with $\ge 75\%$ broadleaved canopy cover ($\ge 10\%$ total cover). |
| **Forest, coniferous** | Woodland with $\ge 75\%$ coniferous canopy cover ($\ge 10\%$ total cover). |
| **Forest, mixed** | Mixed tree canopy ($> 25\%$ of each broadleaved and coniferous). |
| **Shrubland** | Vegetation dominated by shrubs and dwarf woody plants. |
| **Grassland** | Natural and managed pastures, meadows, and herbaceous cover. |
| **Barren** | Bare soil, exposed rock, scree, and sparsely vegetated land. |
| **Wetland** | Inland and coastal peatlands, marshes, and bogs. |
| **Water** | Inland water bodies, rivers, lakes, and reservoirs. |
| **Snow/ice** | Glaciers and perennial snow fields. |

The harmonized version of LUCAS by d'Andrimont et al. (2020) covering field surveys from 2006 to 2018 is [available in Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/JRC_LUCAS_HARMO_THLOC_V1#description). In a first step, we processed this ee.FeatureCollection to match the pre-processing and categorization described in Pflugmacher et al. (2019) and uploaded the resulting ee.FeatureCollection to a publicly accessible Asset (see [LUCAS_1_preprocess.py](habitat-type-mapping/LUCAS_1_preprocess.py) for details).

The resulting standardised reference points (505,509 observations across Europe) were ingested into Earth Engine (`projects/eexnill/assets/geeo_public/LUCAS_HARMO_V1_EO_LC`):

In [ ]:
lucas = ee.FeatureCollection('projects/eexnill/assets/geeo_public/LUCAS_HARMO_V1_EO_LC')
lucas.size()

We can display the pan-European point distribution:

In [ ]:
# names for legend
lc_names = [
    "Artificial land",
    "Cropland, seasonal",
    "Cropland, perennial",
    "Forest, broadleaved",
    "Forest, coniferous",
    "Forest, mixed",
    "Shrubland",
    "Grassland",
    "Barren",
    "Wetland",
    "Water",
    "Snow/ice"
]
# colours for classes
lc_colors = [
    "FF0000",  # 0 - Artificial land (red)
    "FFD700",  # 1 - Cropland, seasonal (golden/yellow)
    "CD853F",  # 2 - Cropland, perennial (tan/brown)
    "32CD32",  # 3 - Forest, broadleaved (forest green)
    "006400",  # 4 - Forest, coniferous (dark green)
    "228B22",  # 5 - Forest, mixed (lime green)
    "FFA500",  # 6 - Shrubland (orange/kaki)
    "9ACD32",  # 7 - Grassland (yellow green)
    "D2B48C",  # 8 - Barren (tan)
    "800080",  # 9 - Wetland (purple)
    "0000FF",  # 10 - Water (blue)
    "FFFFFF"   # 11 - Snow/ice (white)
]

styled_lucas = geemap.ee_vector_style(
    lucas, 
    column="LC_ID", 
    labels=list(range(len(lc_names))), 
    color=lc_colors,
    pointSize=1
)

M = geemap.Map(center=[44, 7], zoom=6)
M.add_basemap('HYBRID')
M.addLayer(styled_lucas, {}, "lc1")
M.add_legend(title="Land Cover Classes", keys=[str(l) for l in lc_names], colors=lc_colors)
M

This massive European-wide dataset holding land cover information for 2015 and 2019 builds the baseline for our example. For all of these points we would like to extract Landsat image time series for the reference period to be able to derive image features suitable for predicting land cover.

Since the dataset spans a large geographic extent and we want to map land cover at relatively fine pixel grain (30m), we need to spatially organise the export also for computational reasons.  

A key step we want to highlight in this example is how users can use *geeo*'s tiling function to request exports in spatial tiles (i.e. subsets). When exporting imagery or tables, it is often usefull to organise the data according to a spatial tiling scheme. 

#### Spatial Tiling
**geeo has in-built functions for creating a spatial tiling scheme for a given CRS.** This faciliates spatial organization of the data similar to the datacube concept and allows precisely aligning output images onto a grid scheme. Especially when working with long and/or dense time series, as well as when working with different geospatial raster datasets in a given region, spatial tiling schemes can be extremely usefull.

A spatial tiling scheme conceptually looks as follows:

<div>
<img src="https://raw.githubusercontent.com/leonsnill/geeo/master/geeo/data/fig/datacube.svg" width="40%" style="display:block; margin: 0 auto;"/>
</div>

- **Grid:** The spatial, regular division of the space into tiles starting the grid origin.
- **Grid Origin:** Upper left corner coordinate where numbering of tiles start in X and Y direction
- **Tile:** A (square) grid cell covering an area defined by the tile size. Associated with a unique identifier (X000_Y0000).
- **Tile Size:** The n times n size in meters covered by a tile.

**geeo contains a modified version of the Global LANd Cover mapping and Estimation (GLANCE) projection and tiling system** (https://github.com/measures-glance/glance-grids). 
GLANCE grids is a tile gridding system that uses Lambert Azimuthal Equal Area projections for different 'continents' to minimize distortion for each region. This way, the GLANCE system allows global application but with regional-continental precise equal area projections. Originally, the grid system is based on a 30m pixel (designed for Landsat) and with tiles that are 5000x5000 pixels in size, i.e. 150x150km tiles. *geeo* allows to specify other tile sizes that all cover the common pixel sizes of medium resolution satellite data (10m, 20m, 30m). 

<br>

<div style="text-align:center;">
    <img src="https://raw.githubusercontent.com/leonsnill/geeo/master/geeo/data/GLANCE-tiles/glance_continents_overview.png" width="40%"/>
    <p><em>Figure: GLANCE grids overview (source: https://measures-glance.github.io/glance-grids/)</em></p>
</div>


For this example, we create a 300 x 300 km GLANCE tiling scheme for Europe:

In [ ]:
glance_eu = geeo.create_glance_tiles(continent_code='EU', tile_size=300000, land_mask=True)

M.add_gdf(glance_eu, style={'color': 'red'}, layer_name="EU Glance", zoom_to_layer=False)
M

**With the reference samples in place and the tiling scheme prepared, we can request the export of the image data.**

When exporting imagery or tables, we can directly specify in the parameter file that we wish to organise the export according to a specified tiling scheme. The parameter `ROI_TILES` allows us to to specify the tiling either based on the `ROI` provided, using the GLANCE tiling-scheme, or using an additional geospatial dataset (local or server-side ee.FeatureCollection).

For our specific example, in order to export the Landsat-7, -8, and -9 Level-2 Time Series Stack (TSS) data at all 505,509 reference points for 2014–2019, we specify our samples `LUCAS_HARMO_V1_EO_LC` as `ROI` and set `ROI_TILES: EU_300`. The parameter specification then looks as follows:

In [ ]:
# load parameters from the .yml file into a dictionary
prm_reference_data_to_table = geeo.load_parameters('https://raw.githubusercontent.com/leonsnill/geeo/develop/docs/tutorials/habitat-type-mapping/example-3_reference-data-to-table.yml')

# we will print the parameters and associated values for illustration
new_parameters_of_interest = [
    'YEAR_MIN', 
    'YEAR_MAX',
    'ROI',
    'ROI_TILES',
    'SENSORS',
    'FEATURES',
    'EXPORT_TABLE',
    'EXPORT_TSS',
    'EXPORT_DESC',
    'EXPORT_TABLE_TILE_SCALE'
]

for p in new_parameters_of_interest:
    print(f"{p}: {prm_reference_data_to_table[p]}")

> **Note:** EXPORT_TABLE is set to False above to avoid triggering extensive export tasks. The exported asset from this run has been precomputed and made publicly accessible for inspection below. 

And as previously, we can simply request the processing and export by calling:

In [ ]:
#run_reference_data_to_table = geeo.run_param(prm_reference_data_to_table)  # commented out to avoid evaluating each tile in a loop

This now automatically triggers the export of the Time Series Stack (TSS) data per 300km tile at the enclosed reference samples as a table. 

Exporting raw analysis-ready observations (Time Series Stack) into tabular format (Parquet/CSV) rather than pre-aggregated metrics also has a few practical advantages:
- **Computational efficiency:** It "only" creates analysis ready data using the level-2 processing of *geeo*, so it does not trigger more "heavy" computations for thousands of points spanning entire Europe at 30m resolution
- **Flexibility:** Once extracted, the tabular Time Series Stack data can be used in local analyses to derive features (e.g. Spectral Temporal Metrics) and only query the features needed for mapping in Earth Engine

Once the Time Series Stack data has been exported for all the points and tiles, we combined them into a single table (i.e., server-side a ee.FeatureCollection) as outlined in the Python file [LUCAS_2_merge-TSS.py](habitat-type-mapping/LUCAS_2_merge-TSS.py).

The merged dataset containing the Time Series Stack data for all the LUCAS samples are accessible here:

In [ ]:
lucas_tss = ee.FeatureCollection('projects/eexnill/assets/geeo_public/landsat-tss/TSS_LUCAS_HARMO_V1_EO_LC_EU')
lucas_tss.first()

This dataset now contains for all the points also the measured Landsat surface reflectance and indices available between 2014-2019.

> **Note:** For completeness but not necessary to continue: the processed reference table containing the engineered features is available for download on [Google Drive](https://drive.google.com/file/d/1_13WRRcbWkTQCML06SjqjmpIeEQZLbPe/view?usp=sharing).

### 2) Local conversion of the image data to image predictors, as well as building and evaluating a model for land cover mapping

With the Time Series Stack retrieved as .csv table, we continued 'offline' and ...

- ... computed candidate Spectral Temporal Metrics across annual and seasonal windows using local scripts (see [LUCAS_3_TSS-to-STM.py](habitat-type-mapping/LUCAS_3_TSS-to-STM.py) for details). 
- ... used the Spectral Temporal Metrics and trained a Random Forest classifiers (see [LUCAS_4_RF-model.py](habitat-type-mapping/LUCAS_4_RF-model.py) for details), reducing the pool of image predictors down to 38 most informative features
- ... uploaded the Random Forest model as Earth Engine compatible Asset to be used for server-side land cover mapping (see [LUCAS_4_RF-model.py](habitat-type-mapping/LUCAS_4_RF-model.py) for details)

With this in place we now can use *geeo* to retrieve the 38 selected image features and the server-side model to predict land cover.

### 3) Server-Side Land Cover Mapping

As just described, the local scikit-learn Random Forest model was uploaded into an Earth Engine asset:

In [ ]:
rf_model_asset = 'projects/eexnill/assets/geeo_public/RF-MODEL_TSS_LUCAS_NTREE-50' 

# geeo has a function to load the table asset to a model
from geeo.level4.model import dt_table_asset_to_model
rf_model = dt_table_asset_to_model(rf_model_asset)

The classifier expects the selected features used during offline model building, these are:

In [ ]:
selected_features = [
    # Full-season metrics
    'full_RED_p75',
    'full_NIR_p25', 'full_NIR_p95', 'full_NIR_stdDev',
    'full_SW2_stdDev',
    'full_MDWI_p75',
    'full_NBR_p5', 'full_NBR_p75', 'full_NBR_p95', 'full_NBR_stdDev',
    'full_NDVI_p25', 'full_NDVI_stdDev',

    # Winter (December to February)
    'winter_MDWI_p75', 'winter_NBR_p5', 'winter_NDVI_p95',

    # Spring (March to May)
    'spring_NIR_p95', 'spring_NIR_stdDev', 'spring_SW1_p95',
    'spring_MDWI_p25', 'spring_MDWI_p50', 'spring_MDWI_p75',
    'spring_NBR_p5', 'spring_NBR_p95',
    'spring_NDVI_p5', 'spring_NDVI_p75',

    # Summer (June to August)
    'summer_RED_stdDev', 'summer_NIR_p95', 'summer_NIR_stdDev',
    'summer_SW1_p95', 'summer_SW1_stdDev', 'summer_MDWI_p95',
    'summer_NBR_p5', 'summer_NBR_stdDev',

    # Autumn (September to November)
    'autumn_NIR_p95', 'autumn_SW1_p95', 'autumn_MDWI_p75',
    'autumn_NBR_p25', 'autumn_NDVI_p25',
]

We can create an ee.Image containing these features using *geeo*:

*1. Full-Season (Annual) STMs:*

In [ ]:
# we create a tiling scheme for a new study area of size 600km x 600km
glance_600 = geeo.create_glance_tiles(continent_code='EU', tile_size=600000, land_mask=True)
tile = 'EU_600-X007-Y007'
roi = glance_600[glance_600['ID600'] == tile]

# --------------------------------------------------------------------------
# FULL SEASON (Jan-Dec)
# we setup the parameters to calculate the STMs for our new study area for 2018
prm_lc_mapping = {
    'YEAR_MIN': 2017,  # mapping land cover for 2018 +- 1 yeasr of data
    'YEAR_MAX': 2019,
    'ROI': roi,
    'SENSORS': ['L7', 'L8', 'L9'],
    'MASKS_LANDSAT': ['cloud', 'cshadow', 'fill', 'dilated'],
    'FEATURES': ['RED', 'NIR', 'SW1', 'SW2', 'NDVI', 'NBR', 'MDWI'],
    'FOLD_CUSTOM': {'month': ['3-5', '6-8', '9-11', '12-2']},  # spring, summer, autumn, winter
    'STM': ['p5', 'p25', 'p50', 'p75', 'p95', 'stdDev'],
    'STM_FOLDING': False
}

# run
full_run_lc = geeo.run_param(prm_lc_mapping)
# get STM and rename bands to match RF model
full_STM = full_run_lc['STM'] 
initial_bandnames = full_STM.bandNames().getInfo()  # get band names
new_bandnames = [f'full_{b}' if not b.startswith('full_') else b for b in initial_bandnames]  # adjust to match the RF model
full_STM = full_STM.multiply(10000).rename(new_bandnames)  # rescale and rename bands


*2. Seasonal Fold STMs:*

In [ ]:
prm_lc_mapping.update({'STM_FOLDING': True})  # now weset folding to True to get seasonal metrics

# run
seasonal_run_lc = geeo.run_param(prm_lc_mapping)
seasonal_STM = seasonal_run_lc['STM']  # get STM

# rename bands to match RF model
season_prefixes = ['spring', 'summer', 'autumn', 'winter']

def set_season_index(img, idx):
    return img.set('system:index', season_prefixes[idx])

images = seasonal_STM.toList(seasonal_STM.size())
seasonal_STM = ee.ImageCollection([
    set_season_index(ee.Image(images.get(i)), i)
    for i in range(4)
])

seasonal_STM = seasonal_STM.toBands().multiply(10000)  # scale by 10,000 to match int16 stored in model

We assemble the 38 selected image predictors into a single multi-band ee.Image ...

In [ ]:
img = ee.Image(full_STM.addBands(seasonal_STM)).select(selected_features)
img

... and once created, we can apply the classifier using Earth Engine's native classify() method:

In [ ]:
# predict land cover
land_cover = img.classify(rf_model)
land_cover = land_cover.toUint8()  # cast to uint8 to save space
land_cover

> **Note:** For such a large area (600km x 600km), three years of Landsat data (2017-2019), seasonal Spectral Temporal Metrics, and a running a classification prediction on top, we have to usually export the result to not exceed available memory. Note also, that such a multi-step pipeline can cost a considerable amount of EECU seconds (and Google limits available compute to users accoridng to different Tiers). Accordingly, we export the image and visualize the result:

In [ ]:
do_export = False  # see note above!

if do_export:
    geeo.export_img(img=land_cover, out_location='Asset', outname='LUCAS_LC_2017-2019_RF_EU_600-X007-Y007',
                    out_dir='projects/eexnill/assets/geeo_public',
                    px_res=30, crs='EU', region=roi, pyramidingPolicy={'.default': 'mode'}, dtype='uint8')

Visualizing the land cover map:

In [ ]:
cl_img = ee.Image('projects/eexnill/assets/geeo_public/LUCAS_LC_2017-2019_RF_EU_600-X007-Y007')

vis_params_cl = {
    'min': 0,
    'max': 11,
    'palette': lc_colors,
    'bands': ['classification']
}
vis_params = {'color': 'red', 'width': 2, 'lineType': 'solid','fillColor': '00000000'}

M = geemap.Map(center=[44.25, 7], zoom=7)
M.add_basemap('HYBRID')
M.addLayer(cl_img, vis_params_cl, 'Classification')
M

## References

- **d'Andrimont, R., et al. (2020)**. Harmonised LUCAS in-situ land cover and use database for field surveys from 2006 to 2018 in the European Union. *Scientific Data*, 7(1), 352. https://doi.org/10.1038/s41597-020-00675-z
- **Frantz, D. (2019)**. FORCE-Landsat + Sentinel-2 analysis ready data and beyond. *Remote Sensing*, 11, https://doi.org/10.3390/rs11091124
- **Harkort, L., et al. (2025)**. Mapping fractional vegetation cover in Sub-Saharan rangelands using phenological feature spaces. *Remote Sensing of Environment*, 319, 114646. https://doi.org/10.1016/j.rse.2025.114646
- **Kowalski, K., et al. (2020)**. Characterizing spring phenology of temperate broadleaf forests using Landsat and Sentinel-2 time series. *International Journal of Applied Earth Observation and Geoinformation*, 92, 102172. https://doi.org/10.1016/j.jag.2020.102172
- **Nill, L., et al. (2022)**. Arctic shrub expansion revealed by Landsat-derived multitemporal vegetation cover fractions in the Western Canadian Arctic. *Remote Sensing of Environment*, 281, 113228. https://doi.org/10.1016/j.rse.2022.113228
- **Oeser, J., et al. (2019)**. Habitat metrics based on multi-temporal Landsat imagery for mapping large mammal habitat. *Remote Sensing in Ecology and Conservation*, 6(1), 52–69. https://doi.org/10.1002/rse2.122
- **Pereira, H. M., et al. (2013)**. Essential Biodiversity Variables. *Science*, 339(6117), 277–278. https://doi.org/10.1126/science.1229931
- **Pflugmacher, D., et al. (2019)**. Mapping pan-European land cover using Landsat spectral-temporal metrics and the European LUCAS survey. *Remote Sensing of Environment*, 221, 583–595. https://doi.org/10.1016/j.rse.2018.12.001
- **Radeloff, V. C., et al. (2019)**. The Dynamic Habitat Indices (DHIs) from MODIS and global biodiversity. *Remote Sensing of Environment*, 222, 204–214. https://doi.org/10.1016/j.rse.2018.12.009
- **Radeloff, V. C., et al. (2024)**. Need and vision for global medium-resolution Landsat and Sentinel-2 data products. *Remote Sensing of Environment*, 300, 113918. https://doi.org/10.1016/j.rse.2023.113918
- **Razenkova, E., et al. (2025)**. Medium-resolution Dynamic Habitat Indices from Landsat and Sentinel-2 satellite imagery. *Ecological Indicators*, 173, 113367. https://doi.org/10.1016/j.ecolind.2025.113367
- **Rufin, P., et al. (2019)**. Mapping cropping practices on a national scale using intra-annual Landsat time series binning. *Remote Sensing*, 11(3), 232. https://doi.org/10.3390/rs11030232
- **Schwieder et al. (2016)**. Mapping Brazilian savanna vegetation gradients with Landsat time series. *International Journal of Applied Earth Observation and Geoinformation*, 52, https://doi.org/10.1016/j.jag.2016.06.019
- **Tulbure, M. G., et al. (2022)**. Regional matters: On the usefulness of regional land-cover datasets in times of global change. *Remote Sensing in Ecology and Conservation*, 8(3), 272–283. https://doi.org/10.1002/rse2.248

---